# Exporting Traces to New Relic via OTLP

TruLens can export its OpenTelemetry spans and GenAI metrics to any OTLP-compatible backend. This notebook points that export at [New Relic](https://newrelic.com/), which ingests OTLP natively.

## Prerequisites

1. A New Relic account and a [license key](https://docs.newrelic.com/docs/apis/intro-apis/new-relic-api-keys/) for ingest.
2. The TruLens OTLP extra, which installs the OTLP gRPC exporter.

TruLens exports OTLP over gRPC, so use New Relic's gRPC port `4317`, not the HTTP port `4318`. See the [New Relic OTLP documentation](https://docs.newrelic.com/docs/opentelemetry/best-practices/opentelemetry-otlp/) for endpoint and authentication details.

In [ ]:
# !pip install "trulens[otlp]"

## Configure the New Relic Endpoint

New Relic's US OTLP endpoint is `https://otlp.nr-data.net:4317`; the EU endpoint is `https://otlp.eu01.nr-data.net:4317`. Authentication is a header named `api-key` set to your license key.

TruLens reads the standard OpenTelemetry environment variables `OTEL_EXPORTER_OTLP_ENDPOINT` and `OTEL_EXPORTER_OTLP_HEADERS` when no explicit `otlp_endpoint` is given. Set `NEW_RELIC_LICENSE_KEY` in your environment before launching this notebook so the key never appears in the notebook itself.

In [ ]:
import os

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://otlp.nr-data.net:4317"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = (
    "api-key=" + os.environ["NEW_RELIC_LICENSE_KEY"]
)

## Create a TruSession with the OTLP Exporter

Selecting `otel_exporter="otlp"` sends spans and GenAI metrics to the endpoint configured above. You can also pass the endpoint directly with `TruSession(otel_exporter="otlp", otlp_endpoint="https://otlp.nr-data.net:4317")`.

**Note**: OTLP mode replaces the default TruLens database exporter. Spans go to New Relic instead of the TruLens database, so the TruLens dashboard and feedback evaluations that read spans from that database will not see them.

In [ ]:
from trulens.core import TruSession

session = TruSession(otel_exporter="otlp")

## Create an Instrumented App

Let's create a simple RAG-style application with instrumentation.

In [ ]:
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes


class SimpleRAGApp:
    """A simple RAG-style application for demonstration."""

    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL)
    def retrieve(self, query: str) -> list:
        """Retrieve relevant contexts for a query."""
        return [
            "TruLens is an open-source library for evaluating LLM apps.",
            "TruLens exports OTEL spans to any OTLP-compatible backend.",
        ]

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate(self, query: str, contexts: list) -> str:
        """Generate an answer based on retrieved contexts."""
        context_text = " ".join(contexts)
        return f"Based on the context: {context_text[:100]}..."

    @instrument()
    def query(self, question: str) -> str:
        """Main entry point: retrieve contexts and generate an answer."""
        contexts = self.retrieve(question)
        return self.generate(question, contexts)


app = SimpleRAGApp()

## Wrap with TruApp and Record

Recording works exactly as it does with the default exporter; only the destination of the spans changes. `session.force_flush()` blocks until pending trace and metric exports have been delivered, which matters in short-lived processes like this notebook.

In [ ]:
from trulens.apps.app import TruApp

tru_app = TruApp(
    app,
    app_name="NewRelicExampleApp",
    app_version="v1",
)

with tru_app as recording:
    result = app.query("What is TruLens?")
    print(f"Answer: {result}")

session.force_flush()

## What to Look For in New Relic

- **Entity**: TruLens sets the `service.name` resource attribute to `trulens`, so the spans appear under **All entities > Services - OpenTelemetry** as a service named `trulens`.
- **Traces**: open the `trulens` entity's distributed traces, or query the raw spans with NRQL:

  ```sql
  FROM Span SELECT * WHERE service.name = 'trulens' SINCE 30 minutes ago
  ```

  Each recorded invocation produces a record root span with retrieval and generation child spans, carrying TruLens semantic attributes in the `ai.observability.*` namespace.
- **GenAI metrics**: when your app makes real LLM calls, TruLens also exports the `gen_ai.client.token.usage` and `gen_ai.client.operation.duration` histogram metrics. TruLens follows the OpenTelemetry GenAI semantic conventions (`gen_ai.*`), the same attribute family New Relic's [AI monitoring](https://docs.newrelic.com/docs/ai-monitoring/intro-to-ai-monitoring/) is built around.